# Phase G1: Ginza single-UE CFR, beam sweep, and normal/sleep comparison

## 1. Scope and physical assumptions

This notebook reproduces a single-UE point-to-point link from the authoritative
`cfr_mamimo.ipynb` reference, but using the project's own `ArrayConfig` and
codebook infrastructure.

Key physical choices:
- TX: 128-port cross-polarized UPA built from `ArrayConfig(4, 8, 2, 2)`.
  - project pol0 → Sionna cross component 0 (-45°).
  - project pol1 → Sionna cross component 1 (+45°).
- RX: single dual-polarized (VH) isotropic element.
- Scene: Ginza_012 XML at 3.5 GHz, 100 MHz bandwidth, 290 K.
- Material override: `itu_concrete` objects are replaced with ITU concrete
  (thickness 0.5 m, scattering 0.3, XPD 0.3), matching `cfr_mamimo.ipynb`.
- UE: first valid position from the existing `rx_1000.pkl`, falling back to a
  reproducible seed-36 sample only if necessary.
- PathSolver: one formal run with `synthetic_array=True`, seed 36, max_depth 15,
  LOS + specular + diffuse reflection enabled, refraction/diffraction disabled.
- CFR: center frequency only (`frequencies=[0.0]`), no normalization.
- Beam selection: valid PMI mask, DFT codebook (8 vertical × 32 horizontal
  oversampled spatial beams, i2=4; physical array columns remain 8), best
  beam by effective-channel power `sum(|H @ w_sionna|²)`.
- Normal/sleep: same `H`, same selected PMI, sleep applies the right-half
  physical-port muting mask without renormalization.

Absolute SNR/SINR is **not** computed here; link-budget details are deferred to
Phase G1.1.

In [ ]:
## 2. Imports and environment

import os
import pickle
import time
import math

os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')

import numpy as np
import torch
import mitsuba as mi

mi.set_variant('cuda_ad_mono_polarized')

import sionna.rt as rt

from mMIMO_sleep.array_config import ArrayConfig
from mMIMO_sleep.simulation.sionna_array import (
    array_config_to_planar_array,
    weights_to_sionna_precoding,
)
from mMIMO_sleep.codebook.dft import generate_dft_codebook
from mMIMO_sleep.codebook.muting import create_right_half_mask, apply_muting_mask
from mMIMO_sleep.codebook.pmi_mask import (
    create_total_loss_pmi_mask,
    beam_indices_from_mask,
)
from mMIMO_sleep.codebook.pmi import PMI, beam_index_to_pmi

print('Environment ready.')
print(f'Mitsuba variant: {mi.variant()}')
print(f'Torch CUDA available: {torch.cuda.is_available()}')


## 3. `cfr_mamimo.ipynb` parameter audit

From the authoritative `cfr_mamimo.ipynb` (TAP_TCB_Resource/Ginza_012) and the
installed `Lib_xhd.tools.sionna_xhd` source, the exact positional parameter
mappings are:

**TX orientation**
```python
mi.Point3f(0, 15/360 * dr.pi, 0)
```
Because `dr.pi` is the mathematical π, this evaluates to
`15/360 * π = 0.1308996938995747 rad = 7.5°`, **not** 15°.

**`SceneParams("clin", True, True, True, 3.5e9, None, None, clin)`**
| Position | Field | Value in notebook |
|---|---|---|
| 1 | `key` | `"clin"` |
| 2 | `info` | `True` |
| 3 | `show` | `True` |
| 4 | `axes` | `True` |
| 5 | `f` | `3.5e9` Hz |
| 6 | `s` | `None` |
| 7 | `cwin` | `None` |
| 8 | `clin` | `"/home/xhd/.../ginza_1/ginza_1.xml"` → mapped locally |

**`PropagParams(15, True, True, True, False, False, False, False, 36)`**
| Position | Field | Value |
|---|---|---|
| 1 | `max_depth` | 15 |
| 2 | `los` | True |
| 3 | `specular_reflection` | True |
| 4 | `diffuse_reflection` | True |
| 5 | `refraction` | False |
| 6 | `diffraction` | False |
| 7 | `edge_diffraction` | False |
| 8 | `diffraction_lit_region` | False |
| 9 | `seed` | 36 |

**`PathsParams(int(6e6), int(2e6), True)`**
| Position | Field | Value |
|---|---|---|
| 1 | `max_num_paths_per_src` | 6,000,000 |
| 2 | `samples_per_src` | 2,000,000 |
| 3 | `synthetic_array` | True |

**`OFDMParams(288, 100e6, 30e3, 4096, 14, 273, False, False, False, 'tf')`**
| Position | Field | Value |
|---|---|---|
| 1 | `CP_length` | 288 |
| 2 | `bandwidth` | 100 MHz |
| 3 | `subcarrier_spacing` | 30 kHz |
| 4 | `num_subcarriers` | 4096 |
| 5 | `num_time_steps` | 14 |
| 6 | `num_resource_blocks` | 273 |
| 7 | `normalize_delays` | False |
| 8 | `normalize` | False |
| 9 | `reverse_direction` | False |
| 10 | `out_type` | `'tf'` |

For this Phase-G1 notebook we only need the carrier frequency and bandwidth;
the OFDM parameters are recorded for Phase G1.1.

In [ ]:
## 4. Ginza configuration

# Authoritative values from cfr_mamimo.ipynb, with the scene path remapped to
# this environment.
scene_xml_path = (
    '/workspace/Study_Sionna/Projects/TAP_TCB_Resource/Ginza_012/'
    'ginza_1/ginza_1.xml'
)
carrier_frequency_hz = 3.5e9
scene_bandwidth_hz = 100e6
temperature_k = 290.0

tx_position = (-122.0, -108.5, 41.0)
# Original notebook expression: mi.Point3f(0, 15/360 * dr.pi, 0)
# Evaluated numerically: 15/360 * pi = 0.1308996938995747 rad = 7.5 deg.
tx_orientation = (0.0, (15.0 / 360.0) * math.pi, 0.0)

tx_power_dbm = 51.13      # recorded, not used for absolute SNR
noise_figure_db = 5.0     # recorded, not used for absolute SNR
seed = 36
max_depth = 15

# PathSolver sample budget for the single-UE formal run.
# The reference uses 2e6 samples per source for 3000 UEs; for one UE we reduce
# this to 1e6 to save time while keeping enough paths.
samples_per_src = 1_000_000
max_num_paths_per_src = 1_000_000

print('Ginza configuration:')
print(f'  scene_xml_path = {scene_xml_path}')
print(f'  carrier_frequency_hz = {carrier_frequency_hz}')
print(f'  scene_bandwidth_hz = {scene_bandwidth_hz}')
print(f'  temperature_k = {temperature_k}')
print(f'  tx_position = {tx_position}')
print(f'  tx_orientation (rad) = {tx_orientation}')
print(f'  tx_orientation y-deg = {math.degrees(tx_orientation[1])}')
print(f'  tx_power_dBm = {tx_power_dbm}')
print(f'  noise_figure_dB = {noise_figure_db}')
print(f'  seed = {seed}')
print(f'  max_depth = {max_depth}')
print(f'  samples_per_src = {samples_per_src}')
print(f'  max_num_paths_per_src = {max_num_paths_per_src}')


In [ ]:
## 5. Scene and material loading

print('Loading Ginza scene...')
scene = rt.load_scene(scene_xml_path)

scene.frequency = mi.Float(carrier_frequency_hz)
scene.bandwidth = mi.Float(scene_bandwidth_hz)
scene.temperature = mi.Float(temperature_k)

# Reproduce the material override from cfr_mamimo.ipynb Cell 2.
my_concrete = rt.ITURadioMaterial(
    name='my_concrete',
    itu_type='concrete',
    thickness=mi.Float(0.5),
    scattering_coefficient=0.3,
    xpd_coefficient=0.3,
)

replaced_objects = []
for obj_name, obj in scene.objects.items():
    mat_name = obj.radio_material.name
    if mat_name in ('itu_concrete', 'concrete'):
        obj.radio_material = my_concrete
        replaced_objects.append((obj_name, mat_name))

if not replaced_objects:
    raise RuntimeError(
        'No scene object had material "itu_concrete" or "concrete"; '
        'the material override from cfr_mamimo.ipynb cannot be applied.'
    )

print('Applied my_concrete override to the following objects:')
for obj_name, old_mat in replaced_objects:
    print(f'  {obj_name}: {old_mat} -> my_concrete')

print(f'Total objects in scene: {len(scene.objects)}')
print(f'Number of replaced concrete objects: {len(replaced_objects)}')


In [ ]:
## 6. TX/RX array configuration

config = ArrayConfig(
    num_subarray_rows=4,
    num_horizontal=8,
    elements_per_subarray=2,
    num_polarizations=2,
)

tx_array = array_config_to_planar_array(
    config,
    pattern='tr38901',
    polarization='cross',
    vertical_spacing=0.5,
    horizontal_spacing=0.5,
)

assert tx_array.array_size == 64
assert tx_array.num_ant == 128

rx_array = rt.PlanarArray(
    num_rows=1,
    num_cols=1,
    pattern='iso',
    polarization='VH',
)

assert rx_array.num_ant == 2

scene.tx_array = tx_array
scene.rx_array = rx_array

scene.add(rt.Transmitter(name='tx', position=tx_position, orientation=tx_orientation))

print('TX/RX arrays configured:')
print(f'  TX array_size = {tx_array.array_size}')
print(f'  TX num_ant = {tx_array.num_ant}')
print(f'  RX num_ant = {rx_array.num_ant}')
print(f'  project pol0 -> Sionna cross component 0 (-45 deg)')
print(f'  project pol1 -> Sionna cross component 1 (+45 deg)')


## 6b. Physical array parameters vs. codebook beam parameters

The physical TX array topology (`ArrayConfig`) and the DFT codebook's angular
oversampling are **independent, separately configured quantities**. They must
not be conflated:

- `config.num_horizontal = 8` is the number of **physical horizontal element
  columns** in the TX subarray grid (fixed by the antenna hardware / ArrayConfig).
- `NUM_HORIZONTAL_BEAMS = 32` is the number of **horizontal DFT codebook
  beams** (i.e. the number of distinct `i11` values). This is a codebook
  design choice, independent of the physical column count.
- horizontal oversampling factor = `NUM_HORIZONTAL_BEAMS / config.num_horizontal`
  = `32 / 8 = 4`.
- `NUM_VERTICAL_BEAMS = 8` (i12 values) and `NUM_I2 = 4` (co-phasing values)
  are unchanged from the physical vertical subarray count and the standard
  4-point co-phasing grid, respectively.

`config.num_horizontal` must **not** be used to set `NUM_HORIZONTAL_BEAMS`;
they serve different roles in `generate_dft_codebook`
(`config.num_horizontal` sets the physical steering-vector length,
`num_horizontal_beams` sets the number of oversampled DFT directions sampled
over that physical aperture).

In [ ]:
## 6c. Codebook beam-count constants (distinct from physical array topology)

NUM_VERTICAL_BEAMS = 8
NUM_HORIZONTAL_BEAMS = 32
NUM_I2 = 4

horizontal_oversampling_factor = NUM_HORIZONTAL_BEAMS / config.num_horizontal

print('Physical array topology (ArrayConfig):')
print(f'  config.num_subarray_rows = {config.num_subarray_rows}')
print(f'  config.num_horizontal (physical columns) = {config.num_horizontal}')
print(f'  config.elements_per_subarray = {config.elements_per_subarray}')
print(f'  config.num_polarizations = {config.num_polarizations}')
print()
print('Codebook beam-count parameters (independent of physical topology):')
print(f'  NUM_VERTICAL_BEAMS = {NUM_VERTICAL_BEAMS}')
print(f'  NUM_HORIZONTAL_BEAMS = {NUM_HORIZONTAL_BEAMS}')
print(f'  NUM_I2 = {NUM_I2}')
print(f'  horizontal oversampling factor = {NUM_HORIZONTAL_BEAMS}/{config.num_horizontal} = {horizontal_oversampling_factor}')


In [ ]:
## 7. UE placement

# Prefer a position from the existing cfr_mamimo UE set.
pkl_path = (
    '/workspace/Study_Sionna/Projects/TAP_TCB_Resource/Ginza_012/'
    'Results/rx_1000.pkl'
)

with open(pkl_path, 'rb') as f:
    ue_data = pickle.load(f)

loaded_positions = [np.array(p, dtype=np.float64) for p in ue_data['all_rx_positions']]
print(f'Loaded {len(loaded_positions)} UE positions from {pkl_path}')


def has_valid_los_path(scene, position, solver=None):
    '''Lightweight probe: return True if at least one path is found.'''
    if solver is None:
        solver = rt.PathSolver()
    scene.add(rt.Receiver(name='rx_probe', position=position.tolist()))
    try:
        paths = solver(
            scene,
            max_depth=max_depth,
            samples_per_src=1000,
            synthetic_array=True,
            los=True,
            specular_reflection=True,
            diffuse_reflection=True,
            refraction=False,
            diffraction=False,
            edge_diffraction=False,
            diffraction_lit_region=False,
            seed=seed,
        )
        return bool(np.asarray(paths.valid).any())
    finally:
        scene.remove('rx_probe')


ue_position = None
ue_source = None
ue_index = -1

probe_solver = rt.PathSolver()
for idx, pos in enumerate(loaded_positions):
    if has_valid_los_path(scene, pos, solver=probe_solver):
        ue_position = pos
        ue_source = f'rx_1000.pkl index {idx}'
        ue_index = idx
        break

if ue_position is None:
    # Reproducible fallback: sample inside the same 700x700 m square used by
    # cfr_mamimo until a valid path is found.
    rng = np.random.default_rng(seed)
    half_range = 350.0
    for attempt in range(5000):
        x = tx_position[0] + rng.uniform(-half_range, half_range)
        y = tx_position[1] + rng.uniform(-half_range, half_range)
        pos = np.array([x, y, 1.5], dtype=np.float64)
        if has_valid_los_path(scene, pos, solver=probe_solver):
            ue_position = pos
            ue_source = f'seed-{seed} random sample attempt {attempt}'
            break
    if ue_position is None:
        raise RuntimeError('Could not find any valid UE position.')

scene.add(rt.Receiver(name='rx', position=ue_position.tolist()))

tx_pos_arr = np.array(tx_position, dtype=np.float64)
horizontal_dist = float(np.linalg.norm(ue_position[:2] - tx_pos_arr[:2]))
dist_3d = float(np.linalg.norm(ue_position - tx_pos_arr))

print('UE placement:')
print(f'  position = {ue_position}')
print(f'  source = {ue_source}')
print(f'  TX-UE horizontal distance = {horizontal_dist:.3f} m')
print(f'  TX-UE 3D distance = {dist_3d:.3f} m')


In [ ]:
## 8. Formal PathSolver run

path_solver_kwargs = {
    'max_depth': max_depth,
    'max_num_paths_per_src': max_num_paths_per_src,
    'samples_per_src': samples_per_src,
    'synthetic_array': True,
    'los': True,
    'specular_reflection': True,
    'diffuse_reflection': True,
    'refraction': False,
    'diffraction': False,
    'edge_diffraction': False,
    'diffraction_lit_region': False,
    'seed': seed,
}

print('Formal PathSolver arguments:')
for k, v in path_solver_kwargs.items():
    print(f'  {k} = {v}')

# Timing methodology: `rt.PathSolver()(...)` may only *dispatch* work on the
# Dr.Jit/CUDA backend without fully materializing results. We therefore time
# three separate stages on the *same* `paths` object (no second solver call):
#   1. call/dispatch time: time for the solver call to return.
#   2. materialization time: time to force-read `paths.valid` (and `paths.tau`,
#      `paths.a[0]`), which requires the backend to complete computation.
#   3. total time through CFR materialization (measured in the next cell).
rt_t0 = time.time()
paths = rt.PathSolver()(scene, **path_solver_kwargs)
rt_t_dispatch = time.time()
dispatch_time = rt_t_dispatch - rt_t0

valid_mask = np.asarray(paths.valid)
valid_count = int(valid_mask.sum())
tau = np.asarray(paths.tau)
tau_min = float(tau.min()) if valid_count else float('nan')
tau_max = float(tau.max()) if valid_count else float('nan')
a0 = np.asarray(paths.a[0])
rt_t_materialized = time.time()
materialize_time = rt_t_materialized - rt_t_dispatch
total_time_to_materialized = rt_t_materialized - rt_t0

print(f'PathSolver call/dispatch time: {dispatch_time:.3f} s')
print(f'PathSolver materialization time (valid/tau/a): {materialize_time:.3f} s')
print(f'Total time through path materialization: {total_time_to_materialized:.3f} s')
print(f'valid path count: {valid_count}')
print(f'tau shape: {tau.shape}')
print(f'tau min/max: {tau_min:.3e} / {tau_max:.3e} s')
print(f'paths.a[0] shape: {a0.shape}')

if valid_count == 0:
    raise RuntimeError('No valid paths found; stopping.')


In [ ]:
## 9. CFR and H extraction

cfr_t0 = time.time()
h_freq = paths.cfr(
    frequencies=mi.Float([0.0]),
    normalize=False,
    normalize_delays=False,
    out_type='numpy',
)
H = np.asarray(h_freq).squeeze()
cfr_t1 = time.time()
cfr_time = cfr_t1 - cfr_t0
total_time_through_cfr = cfr_t1 - rt_t0

print(f'Raw CFR shape: {np.asarray(h_freq).shape}')
print(f'H shape after squeeze: {H.shape}')
print(f'CFR materialization time (paths.cfr + squeeze): {cfr_time:.3f} s')
print(f'Total time through CFR materialization (from PathSolver call): {total_time_through_cfr:.3f} s')

assert H.shape == (2, 128), f'Expected H.shape == (2, 128), got {H.shape}'
assert np.isfinite(H).all(), 'H contains non-finite values'

# Convert to torch complex tensor without changing port order.
H_t = torch.from_numpy(H).to(torch.complex64)

print('Final H:')
print(f'  dtype = {H_t.dtype}')
print(f'  shape = {H_t.shape}')
print(f'  finite = {torch.isfinite(H_t).all().item()}')


In [ ]:
## 10. Valid codebook beam sweep

# Uses the codebook beam-count constants defined above (NUM_VERTICAL_BEAMS=8,
# NUM_HORIZONTAL_BEAMS=32, NUM_I2=4), NOT config.num_horizontal.
codebook = generate_dft_codebook(
    config,
    num_vertical_beams=NUM_VERTICAL_BEAMS,
    num_horizontal_beams=NUM_HORIZONTAL_BEAMS,
    num_i2=NUM_I2,
    device='cpu',
)
print(f'Codebook shape: {codebook.shape}')
assert codebook.shape == (1024, 128), f'Expected (1024, 128), got {codebook.shape}'

valid_pmi_mask = create_total_loss_pmi_mask(
    config,
    num_vertical_beams=NUM_VERTICAL_BEAMS,
    num_horizontal_beams=NUM_HORIZONTAL_BEAMS,
    num_i2=NUM_I2,
    device='cpu',
)
print(f'Valid PMI mask shape: {valid_pmi_mask.shape}')
assert valid_pmi_mask.shape == (8, 32), f'Expected (8, 32), got {valid_pmi_mask.shape}'

valid_spatial_pmi_count = int(valid_pmi_mask.sum())
valid_beam_indices = beam_indices_from_mask(valid_pmi_mask, num_i2=NUM_I2)
valid_full_beam_count = len(valid_beam_indices)

assert valid_full_beam_count == valid_spatial_pmi_count * NUM_I2, (
    f'valid_full_beam_count ({valid_full_beam_count}) != '
    f'valid_spatial_pmi_count * NUM_I2 '
    f'({valid_spatial_pmi_count} * {NUM_I2} = {valid_spatial_pmi_count * NUM_I2})'
)

print(f'Valid spatial PMI count: {valid_spatial_pmi_count}')
print(f'Valid full beam count: {valid_full_beam_count}')

# Sweep only the valid full beams (already expanded over i2), converting each
# project-order codeword to Sionna ordering before multiplying with H.
best_score = -float('inf')
best_flat_idx = -1

for flat_idx in valid_beam_indices:
    w_project = codebook[flat_idx]
    real, imag = weights_to_sionna_precoding(
        w_project.unsqueeze(0), config
    )
    w_sionna = torch.complex(real, imag).squeeze(0)
    g = H_t @ w_sionna
    score = torch.sum(torch.abs(g) ** 2).item()
    if score > best_score:
        best_score = score
        best_flat_idx = flat_idx

selected_pmi = beam_index_to_pmi(
    best_flat_idx,
    num_horizontal_beams=NUM_HORIZONTAL_BEAMS,
    num_vertical_beams=NUM_VERTICAL_BEAMS,
    num_i2=NUM_I2,
)

assert 0 <= selected_pmi.i11 < NUM_HORIZONTAL_BEAMS
assert 0 <= selected_pmi.i12 < NUM_VERTICAL_BEAMS
assert 0 <= selected_pmi.i2 < NUM_I2

expected_index = (
    selected_pmi.i12 * NUM_HORIZONTAL_BEAMS + selected_pmi.i11
) * NUM_I2 + selected_pmi.i2
assert best_flat_idx == expected_index, (
    f'best_flat_idx ({best_flat_idx}) != expected_index ({expected_index})'
)

print('Beam sweep result:')
print(f'  valid spatial PMI count = {valid_spatial_pmi_count}')
print(f'  valid full beam count = {valid_full_beam_count}')
print(f'  selected flat beam index = {best_flat_idx}')
print(f'  selected PMI = {selected_pmi}')
print(f'  expected_index (formula check) = {expected_index}')
print(f'  selected score = {best_score}')


In [ ]:
## 11. Same-H normal/sleep comparison

w_normal_project = codebook[best_flat_idx]
right_half_mask = create_right_half_mask(config, device='cpu')
w_sleep_project = apply_muting_mask(w_normal_project, right_half_mask)

norm_normal = torch.sum(torch.abs(w_normal_project) ** 2).item()
norm_sleep = torch.sum(torch.abs(w_sleep_project) ** 2).item()

print('Weight norms:')
print(f'  ||w_normal||^2 = {norm_normal:.6f}')
print(f'  ||w_sleep||^2 = {norm_sleep:.6f}')

real_n, imag_n = weights_to_sionna_precoding(
    w_normal_project.unsqueeze(0), config
)
w_normal_sionna = torch.complex(real_n, imag_n).squeeze(0)

real_s, imag_s = weights_to_sionna_precoding(
    w_sleep_project.unsqueeze(0), config
)
w_sleep_sionna = torch.complex(real_s, imag_s).squeeze(0)

g_normal = H_t @ w_normal_sionna
g_sleep = H_t @ w_sleep_sionna

power_normal = torch.sum(torch.abs(g_normal) ** 2).item()
power_sleep = torch.sum(torch.abs(g_sleep) ** 2).item()
loss_db = 10.0 * math.log10(power_normal / power_sleep)

print('Effective-channel powers (same H):')
print(f'  power_normal = {power_normal}')
print(f'  power_sleep = {power_sleep}')
print(f'  same-H loss = {loss_db:.4f} dB')


In [ ]:
## 12. Summary assertions

# Verify consistency of the complete pipeline.
assert H_t.shape == (2, 128)
assert torch.isfinite(H_t).all()
assert valid_spatial_pmi_count > 0
assert 0 <= best_flat_idx < codebook.shape[0]
assert abs(norm_normal - 1.0) < 1e-4
assert abs(norm_sleep - 0.5) < 1e-4

# PMI range checks (codebook beam-count constants, not physical topology).
assert 0 <= selected_pmi.i11 < NUM_HORIZONTAL_BEAMS
assert 0 <= selected_pmi.i12 < NUM_VERTICAL_BEAMS
assert 0 <= selected_pmi.i2 < NUM_I2

# Re-verify the flat-index formula independently of cell 10.
expected_index_check = (
    selected_pmi.i12 * NUM_HORIZONTAL_BEAMS + selected_pmi.i11
) * NUM_I2 + selected_pmi.i2
assert best_flat_idx == expected_index_check

# The selected normal power must match the beam-sweep best score.
score_normal = torch.sum(torch.abs(H_t @ w_normal_sionna) ** 2).item()
assert abs(score_normal - best_score) < 1e-5 * max(best_score, 1.0)

# Normal and sleep must use the same PMI.
assert torch.equal(w_normal_project, codebook[best_flat_idx])
assert torch.equal(w_sleep_project, apply_muting_mask(w_normal_project, right_half_mask))

# Powers must be finite and positive.
assert math.isfinite(power_normal) and power_normal > 0
assert math.isfinite(power_sleep) and power_sleep > 0
assert math.isfinite(loss_db)

print('ALL SUMMARY ASSERTIONS PASSED')
print(f'  NUM_VERTICAL_BEAMS={NUM_VERTICAL_BEAMS}, NUM_HORIZONTAL_BEAMS={NUM_HORIZONTAL_BEAMS}, NUM_I2={NUM_I2}')
print(f'  Codebook shape: {tuple(codebook.shape)}')
print(f'  Valid PMI mask shape: {tuple(valid_pmi_mask.shape)}')
print(f'  Valid spatial PMI count: {valid_spatial_pmi_count}')
print(f'  Valid full beam count: {valid_full_beam_count}')
print(f'  Selected flat beam index: {best_flat_idx}')
print(f'  Selected PMI: i11={selected_pmi.i11}, i12={selected_pmi.i12}, i2={selected_pmi.i2}')
print(f'  Normal power: {power_normal}')
print(f'  Sleep power: {power_sleep}')
print(f'  Same-H loss: {loss_db:.4f} dB')


## 12b. Why the Phase-G1-fix result matches the superseded 8×8 result

The Phase-G1-fix beam sweep (`NUM_HORIZONTAL_BEAMS = 32`) selected
`PMI(i11=24, i12=1, i2=0)`, while the superseded Phase-G1 sweep (which
incorrectly used `NUM_HORIZONTAL_BEAMS = 8`, conflating it with the physical
column count `config.num_horizontal`) selected `PMI(i11=6, i12=1, i2=0)`.

The near-identical `power_normal`, `power_sleep`, and `same-H loss` values
between these two runs are **not a numerical coincidence** and **not caused
by reusing the old result**. They arise because both PMIs correspond to the
same normalized horizontal DFT spatial frequency, i.e. the same physical
steering direction:

```
i11_old / Kh_old = 6 / 8  = 0.75
i11_new / Kh_new = 24 / 32 = 0.75
```

A horizontal DFT beam's steering direction is determined only by the ratio
`i11 / num_horizontal_beams` (the fractional cycles per physical element in
`dft_vector`), not by `i11` or `num_horizontal_beams` individually. Since
`6/8` and `24/32` are the same fraction, the two PMIs steer the physical
8-column aperture toward the same direction, producing the same array
response and therefore the same effective-channel power and same-H loss.
This is expected: the 32-beam codebook is a 4x oversampled refinement of the
8-beam grid, and `i11=24` is exactly the oversampled-grid beam that coincides
with the old `i11=6` on the coarser grid.

## 13. Deferred link-budget items (Phase G1.1)

The following quantities are recorded but **not** used for absolute SNR/SINR in
this notebook:

- TX power: `tx_power_dbm = 51.13` dBm.
- Noise figure: `noise_figure_db = 5.0` dB.
- OFDM configuration from `cfr_mamimo.ipynb`:
  - CP length = 288 samples
  - Bandwidth = 100 MHz
  - Subcarrier spacing = 30 kHz
  - FFT size / subcarriers = 4096
  - Time steps per slot = 14
  - Number of resource blocks = 273

To compute absolute SNR/SINR in Phase G1.1, the following must be resolved:

1. Active subcarrier count / resource-block allocation.
2. Per-subcarrier / per-RB power allocation from 51.13 dBm total TX power.
3. Thermal noise per 30 kHz subcarrier: `k * T * 30 kHz` with `T = 290 K` and
   `NF = 5 dB`.
4. Relationship between the single center-frequency CFR used here and the full
   OFDM bandwidth used in the reference notebook.
5. Whether the reference's `sigma2_dBm = -174 + 10*log10(30e3) + 5` formula
   should be applied per-subcarrier or aggregated.

## 14. Phase G1.1: Explicit link-budget assumptions

Phase G1.1 adds a wideband CFR and an absolute per-subcarrier SNR/SINR
calculation. Phase G1.1A's read-only audit established that the old
`cfr_mamimo.ipynb` **never defined a precise power convention for
`51.13 dBm`, and never multiplied it into the CFR**: `Transmitter.power_dbm`
is stored on the Sionna RT `Transmitter` object as metadata only, and is not
consumed anywhere in `PathSolver` / `FieldCalculator` / `Paths.cir` /
`Paths.cfr` (verified against the installed sionna-rt 1.2.2 source).

Because no such convention exists in the reference code, Phase G1.1
introduces the following as **new project assumptions**, not as facts
recovered from `cfr_mamimo.ipynb`:

1. `51.13 dBm` is defined as the **normal-mode total conducted TX power**
   (summed across all 128 physical TX ports).
2. This total power is **shared equally over the 3276 active subcarriers**
   (`P_SC_W = P_TX_NORMAL_W / 3276`), independent of resource-block
   boundaries.
3. The unit-norm normal codeword (`sum(|w_normal|^2) = 1`) distributes this
   per-subcarrier power across TX ports; no additional per-port scaling is
   applied beyond the codeword itself.
4. Sleep mode uses the **same per-subcarrier scalar power** `P_SC_W`, but an
   **unnormalized, muted codeword** (`sum(|w_sleep|^2) = 0.5`). The codeword
   norm itself is what reduces the actual radiated power in sleep mode
   (`P_TX_SLEEP_W = P_TX_NORMAL_W * 0.5`, Section 19); no separate
   normalization step is applied to `w_sleep`.
5. No interference is modeled in this phase (single UE, single TX): SINR is
   defined to equal SNR.
6. Cyclic-prefix (CP) overhead is excluded from the instantaneous
   per-subcarrier SNR calculation. CP efficiency is reported in Section 15
   purely for reference and is **not** applied as a throughput or power
   derating factor in this notebook.

These are explicit modeling choices for Phase G1.1 and may be revisited in
later phases.

### 14b. Two distinct kinds of "average": per-subcarrier dB mean vs. linear-domain (wideband) mean

Because `log` is nonlinear, this notebook deliberately reports **two
different, non-interchangeable** summary statistics for both received power
and SNR, and never conflates them under a single name:

- **`mean_of_subcarrier_rx_power_dbm` / `mean_of_subcarrier_snr_db`**:
  `mean(10*log10(x))`, i.e. the arithmetic mean taken *after* converting each
  subcarrier's linear-domain value to dB. This is the mean of the per-subcarrier
  dB curve plotted in Section 23.
- **`wideband_average_rx_power_dbm` / `wideband_average_snr_db`**:
  `10*log10(mean(x))`, i.e. the linear-domain (wideband) average taken
  *before* converting to dB. This equals the dB value of the single
  wideband-averaged linear quantity (`wideband_average_rx_power_w`,
  `wideband_average_snr_linear`).

In general `mean(10*log10(x)) != 10*log10(mean(x))` for any non-degenerate
distribution of `x`, by Jensen's inequality (`log` is concave, so the
dB-domain mean is always `<=` the linear-domain-mean-then-dB value). Both
notebook and consumers of this notebook must **never** use these two
quantities interchangeably; they are computed and reported as clearly
separate, distinctly-named statistics for both normal and sleep in
Section 22.

In [ ]:
## 15. OFDM configuration (Phase G1.1)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

FFT_SIZE = 4096
CP_LENGTH = 288
SUBCARRIER_SPACING_HZ = 30e3
NUM_RESOURCE_BLOCKS = 273
SUBCARRIERS_PER_RB = 12
NUM_ACTIVE_SUBCARRIERS = 3276
LEFT_GUARD = 410
RIGHT_GUARD = 410

assert FFT_SIZE - LEFT_GUARD - RIGHT_GUARD == NUM_ACTIVE_SUBCARRIERS
assert NUM_RESOURCE_BLOCKS * SUBCARRIERS_PER_RB == NUM_ACTIVE_SUBCARRIERS

fft_sample_rate_hz = FFT_SIZE * SUBCARRIER_SPACING_HZ
occupied_bandwidth_hz = NUM_ACTIVE_SUBCARRIERS * SUBCARRIER_SPACING_HZ
useful_symbol_duration_s = 1.0 / SUBCARRIER_SPACING_HZ
cp_duration_s = CP_LENGTH / fft_sample_rate_hz
cp_efficiency = FFT_SIZE / (FFT_SIZE + CP_LENGTH)
# Not a "sampling frequency": this is the rate at which full (FFT + CP)
# OFDM symbols repeat, i.e. fft_sample_rate_hz / (FFT_SIZE + CP_LENGTH).
ofdm_symbol_repetition_rate_including_cp_hz = (
    fft_sample_rate_hz / (FFT_SIZE + CP_LENGTH)
)

print('OFDM configuration (Phase G1.1):')
print(f'  FFT sample rate = {fft_sample_rate_hz / 1e6:.2f} MHz')
print(f'  occupied bandwidth = {occupied_bandwidth_hz / 1e6:.2f} MHz')
print(f'  useful symbol duration = {useful_symbol_duration_s * 1e6:.3f} us')
print(f'  CP duration = {cp_duration_s * 1e6:.5f} us')
print(f'  CP efficiency = {FFT_SIZE}/({FFT_SIZE}+{CP_LENGTH}) = {cp_efficiency:.7f}')
print(
    '  OFDM symbol repetition rate including CP = '
    f'{ofdm_symbol_repetition_rate_including_cp_hz / 1e3:.3f} kHz'
)


In [ ]:
## 16. Full and active subcarrier frequency grids

full_frequencies = rt.subcarrier_frequencies(FFT_SIZE, SUBCARRIER_SPACING_HZ)
assert len(full_frequencies) == FFT_SIZE

active_frequencies = full_frequencies[LEFT_GUARD:LEFT_GUARD + NUM_ACTIVE_SUBCARRIERS]
assert len(active_frequencies) == NUM_ACTIVE_SUBCARRIERS

active_frequencies_np = np.asarray(active_frequencies, dtype=np.float64)

assert abs(active_frequencies_np[0] - (-49.14e6)) < 1e3
assert abs(active_frequencies_np[-1] - 49.11e6) < 1e3
assert np.any(np.isclose(active_frequencies_np, 0.0))
spacing = np.diff(active_frequencies_np)
assert np.allclose(spacing, SUBCARRIER_SPACING_HZ)

center_sc_index = int(np.argmin(np.abs(active_frequencies_np)))
assert active_frequencies_np[center_sc_index] == 0.0

print('Frequency grid:')
print(f'  full grid length = {len(full_frequencies)}')
print(f'  active grid length = {len(active_frequencies)}')
print(f'  active_frequencies[0] = {active_frequencies_np[0] / 1e6:.4f} MHz')
print(f'  active_frequencies[-1] = {active_frequencies_np[-1] / 1e6:.4f} MHz')
print(f'  center subcarrier index (0 Hz) = {center_sc_index}')


In [ ]:
## 17. Wideband CFR from the same PathSolver run (no re-run)

# Reuses the `paths` object computed in Section 8 (Formal PathSolver run).
# No second call to PathSolver is made.
wb_cfr_t0 = time.time()
h_wideband_raw = paths.cfr(
    frequencies=active_frequencies,
    normalize=False,
    normalize_delays=False,
    out_type='numpy',
)
wb_cfr_t1 = time.time()
wb_cfr_time = wb_cfr_t1 - wb_cfr_t0

print(f'Raw wideband CFR shape: {np.asarray(h_wideband_raw).shape}')
print(f'Wideband CFR materialization time: {wb_cfr_time:.3f} s')

# Squeeze singleton axes (rx device, tx device, time step) -> (rx_ant, tx_ant, freq)
h_wideband_squeezed = np.asarray(h_wideband_raw).squeeze()
print(f'Squeezed shape: {h_wideband_squeezed.shape}')
assert h_wideband_squeezed.shape == (2, 128, NUM_ACTIVE_SUBCARRIERS)

# Reorder axes to (subcarrier, rx_ant, tx_ant).
H_wideband = np.transpose(h_wideband_squeezed, (2, 0, 1))
assert H_wideband.shape == (NUM_ACTIVE_SUBCARRIERS, 2, 128)
assert np.iscomplexobj(H_wideband)
assert np.isfinite(H_wideband).all()

H_wideband_t = torch.from_numpy(H_wideband).to(torch.complex64)

# Cross-check against the G1 center-frequency H (single-frequency solve on the
# same `paths` object). At f=0 Hz, exp(-j*2*pi*f*tau) == 1 regardless of tau,
# so the two must agree to numerical (complex64) precision, not merely to a
# loosely-chosen absolute tolerance.
H_center_from_wideband = H_wideband[center_sc_index]
H_center_t = torch.from_numpy(H_center_from_wideband).to(torch.complex64)

abs_err = torch.abs(H_center_t - H_t)
center_diff = float(abs_err.max())
h_mag = torch.abs(H_t)
rel_err = abs_err / h_mag.clamp_min(1e-30)
max_rel_err = float(rel_err.max())
h_mag_min = float(h_mag.min())
h_mag_max = float(h_mag.max())

print(f'Max |H_wideband[center] - H (G1 center-freq)| = {center_diff:.3e}')
print(f'Max relative error |H_wideband[center] - H| / |H| = {max_rel_err:.3e}')
print(f'|H| magnitude range = [{h_mag_min:.3e}, {h_mag_max:.3e}]')

# Strict center-frequency consistency check. Both H_center_t and H_t are
# complex64 CFR values computed from the same `paths` object at f=0 Hz, so
# they must agree to within complex64 rounding error. Do NOT widen atol/rtol
# to make this pass; if it fails, the diagnostics above report the actual
# max absolute error, max relative error, and |H| range for investigation.
torch.testing.assert_close(H_center_t, H_t, rtol=1e-5, atol=1e-12)

print('Wideband CFR:')
print(f'  H_wideband.shape = {H_wideband.shape}')
print(f'  dtype = {H_wideband_t.dtype}')
print(f'  all finite = {bool(np.isfinite(H_wideband).all())}')


In [ ]:
## 18. Wideband valid-beam sweep

# Reuses `codebook`, `valid_pmi_mask`, `valid_beam_indices`,
# `valid_spatial_pmi_count`, `valid_full_beam_count` from Section 10 (Phase G1).
# Convert ALL valid beams to Sionna ordering in one batched call.
w_valid_project = codebook[valid_beam_indices]  # (1020, 128)
real_valid, imag_valid = weights_to_sionna_precoding(w_valid_project, config)
w_valid_sionna = torch.complex(real_valid, imag_valid)  # (1020, 128)

num_valid_beams = w_valid_sionna.shape[0]
assert num_valid_beams == valid_full_beam_count

BEAM_CHUNK_SIZE = 128

wideband_scores = torch.empty(num_valid_beams, dtype=torch.float32)

wb_sweep_t0 = time.time()
for start in range(0, num_valid_beams, BEAM_CHUNK_SIZE):
    end = min(start + BEAM_CHUNK_SIZE, num_valid_beams)
    w_chunk = w_valid_sionna[start:end]  # (C, 128)
    # g[k, c, r] = sum_i H_wideband_t[k, r, i] * w_chunk[c, i]
    g_chunk = torch.einsum('kri,ci->kcr', H_wideband_t, w_chunk)  # (K, C, 2)
    per_sc_gain_chunk = torch.sum(torch.abs(g_chunk) ** 2, dim=-1)  # (K, C)
    wideband_scores[start:end] = per_sc_gain_chunk.mean(dim=0)
    del g_chunk, per_sc_gain_chunk
wb_sweep_time = time.time() - wb_sweep_t0

best_pos_wideband = int(torch.argmax(wideband_scores).item())
best_flat_idx_wideband = valid_beam_indices[best_pos_wideband]
best_wideband_score = float(wideband_scores[best_pos_wideband].item())

selected_pmi_wideband = beam_index_to_pmi(
    best_flat_idx_wideband,
    num_horizontal_beams=NUM_HORIZONTAL_BEAMS,
    num_vertical_beams=NUM_VERTICAL_BEAMS,
    num_i2=NUM_I2,
)

assert 0 <= selected_pmi_wideband.i11 < NUM_HORIZONTAL_BEAMS
assert 0 <= selected_pmi_wideband.i12 < NUM_VERTICAL_BEAMS
assert 0 <= selected_pmi_wideband.i2 < NUM_I2

# Recompute per-subcarrier gain for the winning beam only (for reporting).
w_best_wb_project = codebook[best_flat_idx_wideband]
real_best_wb, imag_best_wb = weights_to_sionna_precoding(
    w_best_wb_project.unsqueeze(0), config
)
w_best_wb_sionna = torch.complex(real_best_wb, imag_best_wb).squeeze(0)
g_best_wb = H_wideband_t @ w_best_wb_sionna  # (3276, 2)
per_sc_gain_best_wb = torch.sum(torch.abs(g_best_wb) ** 2, dim=-1).numpy()  # (3276,)

center_gain_wideband = float(per_sc_gain_best_wb[center_sc_index])
min_gain_wideband = float(per_sc_gain_best_wb.min())
median_gain_wideband = float(np.median(per_sc_gain_best_wb))
max_gain_wideband = float(per_sc_gain_best_wb.max())

matches_center_selected = (best_flat_idx_wideband == best_flat_idx)

print('Wideband beam sweep result:')
print(f'  sweep time = {wb_sweep_time:.3f} s')
print(f'  selected flat beam index (wideband) = {best_flat_idx_wideband}')
print(f'  selected PMI (wideband) = {selected_pmi_wideband}')
print(f'  wideband mean channel gain = {best_wideband_score:.6e}')
print(f'  center-subcarrier gain = {center_gain_wideband:.6e}')
print(
    '  min/median/max gain = '
    f'{min_gain_wideband:.6e} / {median_gain_wideband:.6e} / {max_gain_wideband:.6e}'
)
print(
    f'  matches G1 center-selected PMI (flat index {best_flat_idx}, '
    f'{selected_pmi})? {matches_center_selected}'
)
print(
    '  Note: all beams share the same 3276 active subcarriers, so '
    'argmax(mean) == argmax(sum); mean is reported for interpretability.'
)


In [ ]:
## 19. Power model (Phase G1.1 explicit assumption, see Section 14)

P_TX_NORMAL_DBM = 51.13
P_TX_NORMAL_W = 10 ** ((P_TX_NORMAL_DBM - 30) / 10)
P_SC_W = P_TX_NORMAL_W / NUM_ACTIVE_SUBCARRIERS
P_SC_DBM = 30 + 10 * math.log10(P_SC_W)

assert abs(P_TX_NORMAL_W - 129.7179271) < 1e-3
assert abs(P_SC_W - 0.03959644) < 1e-6
assert abs(P_SC_DBM - 15.97656) < 1e-3

print('TX power model (Phase G1.1 assumption):')
print(f'  P_TX_NORMAL_DBM = {P_TX_NORMAL_DBM} dBm')
print(f'  P_TX_NORMAL_W = {P_TX_NORMAL_W:.7f} W')
print(f'  P_SC_W (per active subcarrier) = {P_SC_W:.8f} W')
print(f'  P_SC_DBM = {P_SC_DBM:.5f} dBm')

# Use the wideband-selected PMI (Section 18) for the normal/sleep weight pair.
w_normal_wb_project = codebook[best_flat_idx_wideband]
right_half_mask_wb = create_right_half_mask(config, device='cpu')
w_sleep_wb_project = apply_muting_mask(w_normal_wb_project, right_half_mask_wb)

norm_normal_wb = torch.sum(torch.abs(w_normal_wb_project) ** 2).item()
norm_sleep_wb = torch.sum(torch.abs(w_sleep_wb_project) ** 2).item()

assert abs(norm_normal_wb - 1.0) < 1e-4
assert abs(norm_sleep_wb - 0.5) < 1e-4

# Actual total conducted power in sleep mode: derived from the muted codeword
# norm, not from an independent halving (avoids double-counting the muting
# loss, see Section 14 assumption 4).
P_TX_SLEEP_W = P_TX_NORMAL_W * norm_sleep_wb
P_TX_SLEEP_DBM = 30 + 10 * math.log10(P_TX_SLEEP_W)

assert abs(P_TX_SLEEP_W - 64.85896) < 1e-3
assert abs(P_TX_SLEEP_DBM - 48.1197) < 1e-3

print('Normal/sleep TX weight power:')
print(f'  ||w_normal||^2 = {norm_normal_wb:.6f}')
print(f'  ||w_sleep||^2 = {norm_sleep_wb:.6f}')
print(f'  P_TX_SLEEP_W = {P_TX_SLEEP_W:.5f} W')
print(f'  P_TX_SLEEP_DBM = {P_TX_SLEEP_DBM:.4f} dBm')


In [ ]:
## 20. Noise model

BOLTZMANN = 1.380649e-23
TEMPERATURE_K = 290.0
NOISE_FIGURE_DB = 5.0
NOISE_FACTOR = 10 ** (NOISE_FIGURE_DB / 10)
NOISE_BANDWIDTH_HZ = 30e3

N_SC_W = BOLTZMANN * TEMPERATURE_K * NOISE_BANDWIDTH_HZ * NOISE_FACTOR
N_SC_DBM = 30 + 10 * math.log10(N_SC_W)

# Old cfr_mamimo.ipynb approximation (Phase G1.1A audit), reported for
# cross-check only, not used in the SNR calculation below.
N_SC_OLD_DBM = -174 + 10 * math.log10(30e3) + 5

diff_db = abs(N_SC_DBM - N_SC_OLD_DBM)

assert abs(N_SC_W - 3.7984161e-16) < 1e-19
assert abs(N_SC_DBM - (-124.203975)) < 1e-3
assert diff_db < 0.03

print('Noise model:')
print(f'  N_SC_W (physical formula k*T*B*F) = {N_SC_W:.7e} W')
print(f'  N_SC_DBM (physical formula) = {N_SC_DBM:.6f} dBm')
print(
    '  N_SC_OLD_DBM (cfr_mamimo approximation, -174+10log10(BW)+NF) = '
    f'{N_SC_OLD_DBM:.6f} dBm'
)
print(f'  |difference| = {diff_db:.5f} dB (< 0.03 dB tolerance)')


In [ ]:
## 21. Per-subcarrier MRC gain and SNR (normal/sleep)

real_n_wb, imag_n_wb = weights_to_sionna_precoding(
    w_normal_wb_project.unsqueeze(0), config
)
w_normal_wb_sionna = torch.complex(real_n_wb, imag_n_wb).squeeze(0)

real_s_wb, imag_s_wb = weights_to_sionna_precoding(
    w_sleep_wb_project.unsqueeze(0), config
)
w_sleep_wb_sionna = torch.complex(real_s_wb, imag_s_wb).squeeze(0)

g_normal_wb = H_wideband_t @ w_normal_wb_sionna  # (3276, 2)
g_sleep_wb = H_wideband_t @ w_sleep_wb_sionna    # (3276, 2)

# sum_rx(|g|^2): unit-norm-precoded channel power gain after ideal MRC
# combining across the 2 RX (VH) ports.
gain_normal = torch.sum(torch.abs(g_normal_wb) ** 2, dim=-1).numpy()  # (3276,)
gain_sleep = torch.sum(torch.abs(g_sleep_wb) ** 2, dim=-1).numpy()    # (3276,)

assert np.isfinite(gain_normal).all() and (gain_normal >= 0).all()
assert np.isfinite(gain_sleep).all() and (gain_sleep >= 0).all()

# Received power: per-subcarrier TX scalar power (Section 19) times the
# channel power gain. The same P_SC_W scale applies to both states; the
# sleep-mode power reduction comes entirely from the muted codeword's gain.
P_RX_NORMAL_W = P_SC_W * gain_normal
P_RX_SLEEP_W = P_SC_W * gain_sleep

assert np.isfinite(P_RX_NORMAL_W).all() and (P_RX_NORMAL_W >= 0).all()
assert np.isfinite(P_RX_SLEEP_W).all() and (P_RX_SLEEP_W >= 0).all()

SNR_NORMAL = P_RX_NORMAL_W / N_SC_W
SNR_SLEEP = P_RX_SLEEP_W / N_SC_W

assert np.isfinite(SNR_NORMAL).all() and (SNR_NORMAL >= 0).all()
assert np.isfinite(SNR_SLEEP).all() and (SNR_SLEEP >= 0).all()

# No interference is modeled in this phase (Section 14, assumption 5), so
# SINR is defined to equal SNR. No fabricated interference term is added.
SINR_NORMAL = SNR_NORMAL
SINR_SLEEP = SNR_SLEEP
assert np.array_equal(SINR_NORMAL, SNR_NORMAL)
assert np.array_equal(SINR_SLEEP, SNR_SLEEP)

SNR_NORMAL_DB = 10 * np.log10(SNR_NORMAL)
SNR_SLEEP_DB = 10 * np.log10(SNR_SLEEP)

print('Per-subcarrier MRC gain and SNR (wideband-selected PMI):')
print(f'  gain_normal: min={gain_normal.min():.4e}, max={gain_normal.max():.4e}')
print(f'  gain_sleep:  min={gain_sleep.min():.4e}, max={gain_sleep.max():.4e}')
print(f'  SNR_NORMAL_DB: min={SNR_NORMAL_DB.min():.3f}, max={SNR_NORMAL_DB.max():.3f}')
print(f'  SNR_SLEEP_DB:  min={SNR_SLEEP_DB.min():.3f}, max={SNR_SLEEP_DB.max():.3f}')


In [ ]:
## 22. Link-budget summary statistics (Phase G1.1)

def _pct(arr, q):
    return float(np.percentile(arr, q))

P_RX_NORMAL_DBM = 30 + 10 * np.log10(P_RX_NORMAL_W)
P_RX_SLEEP_DBM = 30 + 10 * np.log10(P_RX_SLEEP_W)

loss_db_per_sc = 10 * np.log10(gain_normal / gain_sleep)

# --- Two non-interchangeable averaging conventions (see Section 14b) ---
# (1) mean_of_subcarrier_*: mean(10*log10(x)), i.e. the dB-domain mean taken
#     *after* converting each subcarrier to dB.
mean_of_subcarrier_rx_power_dbm_normal = float(np.mean(P_RX_NORMAL_DBM))
mean_of_subcarrier_rx_power_dbm_sleep = float(np.mean(P_RX_SLEEP_DBM))
mean_of_subcarrier_snr_db_normal = float(np.mean(SNR_NORMAL_DB))
mean_of_subcarrier_snr_db_sleep = float(np.mean(SNR_SLEEP_DB))

# (2) wideband_average_*: 10*log10(mean(x)), i.e. the linear-domain
#     (wideband) mean taken *before* converting to dB.
wideband_average_rx_power_w = float(np.mean(P_RX_NORMAL_W))
wideband_average_rx_power_dbm = 30 + 10 * math.log10(wideband_average_rx_power_w)
wideband_average_rx_power_w_sleep = float(np.mean(P_RX_SLEEP_W))
wideband_average_rx_power_dbm_sleep = 30 + 10 * math.log10(wideband_average_rx_power_w_sleep)

wideband_average_snr_linear = float(np.mean(SNR_NORMAL))
wideband_average_snr_db = 10 * math.log10(wideband_average_snr_linear)
wideband_average_snr_linear_sleep = float(np.mean(SNR_SLEEP))
wideband_average_snr_db_sleep = 10 * math.log10(wideband_average_snr_linear_sleep)

# mean(10*log10(x)) and 10*log10(mean(x)) must not be conflated: by Jensen's
# inequality (log is concave) the dB-domain mean is always <= the
# linear-domain-mean-then-dB value for non-degenerate x.
assert mean_of_subcarrier_rx_power_dbm_normal <= wideband_average_rx_power_dbm + 1e-9
assert mean_of_subcarrier_rx_power_dbm_sleep <= wideband_average_rx_power_dbm_sleep + 1e-9
assert mean_of_subcarrier_snr_db_normal <= wideband_average_snr_db + 1e-9
assert mean_of_subcarrier_snr_db_sleep <= wideband_average_snr_db_sleep + 1e-9

print('=== TX power ===')
print(f'  Normal total conducted power: {P_TX_NORMAL_W:.5f} W ({P_TX_NORMAL_DBM:.4f} dBm)')
print(f'  Sleep total conducted power:  {P_TX_SLEEP_W:.5f} W ({P_TX_SLEEP_DBM:.4f} dBm)')
print(
    f'  Per-subcarrier TX scalar power (both states): {P_SC_W:.8f} W '
    f'({P_SC_DBM:.5f} dBm)'
)
print()
print('=== Received power: mean_of_subcarrier_rx_power_dbm = mean(10*log10(P_rx)) ===')
print(
    f'  Normal: {mean_of_subcarrier_rx_power_dbm_normal:.3f} dBm, '
    f'median={np.median(P_RX_NORMAL_DBM):.3f}, '
    f'min={np.min(P_RX_NORMAL_DBM):.3f}, max={np.max(P_RX_NORMAL_DBM):.3f}'
)
print(
    f'  Sleep:  {mean_of_subcarrier_rx_power_dbm_sleep:.3f} dBm, '
    f'median={np.median(P_RX_SLEEP_DBM):.3f}, '
    f'min={np.min(P_RX_SLEEP_DBM):.3f}, max={np.max(P_RX_SLEEP_DBM):.3f}'
)
print()
print(
    '=== Received power: wideband_average_rx_power_dbm = '
    '10*log10(mean(P_rx_w)) [linear-domain average] ==='
)
print(
    f'  Normal: wideband_average_rx_power_w={wideband_average_rx_power_w:.6e} W '
    f'-> {wideband_average_rx_power_dbm:.3f} dBm'
)
print(
    f'  Sleep:  wideband_average_rx_power_w={wideband_average_rx_power_w_sleep:.6e} W '
    f'-> {wideband_average_rx_power_dbm_sleep:.3f} dBm'
)
print(
    '  NOTE: mean(10*log10(x)) != 10*log10(mean(x)); these are two distinct, '
    'non-interchangeable statistics (Section 14b).'
)
print()
print('=== SNR: mean_of_subcarrier_snr_db = mean(10*log10(SNR)) ===')
print(
    f'  Normal: {mean_of_subcarrier_snr_db_normal:.3f} dB, '
    f'median={np.median(SNR_NORMAL_DB):.3f}, '
    f'min={np.min(SNR_NORMAL_DB):.3f}, max={np.max(SNR_NORMAL_DB):.3f}'
)
print(
    f'          p5={_pct(SNR_NORMAL_DB, 5):.3f}, p95={_pct(SNR_NORMAL_DB, 95):.3f}'
)
print(
    f'  Sleep:  {mean_of_subcarrier_snr_db_sleep:.3f} dB, '
    f'median={np.median(SNR_SLEEP_DB):.3f}, '
    f'min={np.min(SNR_SLEEP_DB):.3f}, max={np.max(SNR_SLEEP_DB):.3f}'
)
print(
    f'          p5={_pct(SNR_SLEEP_DB, 5):.3f}, p95={_pct(SNR_SLEEP_DB, 95):.3f}'
)
print()
print(
    '=== SNR: wideband_average_snr_db = 10*log10(mean(SNR_linear)) '
    '[linear-domain average] ==='
)
print(
    f'  Normal: wideband_average_snr_linear={wideband_average_snr_linear:.6e} '
    f'-> {wideband_average_snr_db:.3f} dB'
)
print(
    f'  Sleep:  wideband_average_snr_linear={wideband_average_snr_linear_sleep:.6e} '
    f'-> {wideband_average_snr_db_sleep:.3f} dB'
)
print()
print('=== Per-subcarrier normal-sleep channel-power loss (dB) ===')
print(
    f'  mean={np.mean(loss_db_per_sc):.4f}, median={np.median(loss_db_per_sc):.4f}, '
    f'min={np.min(loss_db_per_sc):.4f}, max={np.max(loss_db_per_sc):.4f}'
)
print(
    f'  subcarriers with negative loss (sleep gain > normal gain): '
    f'{int(np.sum(loss_db_per_sc < 0))} / {len(loss_db_per_sc)}'
)


## 22b. Interpreting negative per-subcarrier normal-sleep loss

`loss_db_per_sc` (Section 22) can be **negative** on a minority of
subcarriers, meaning the sleep effective-channel gain exceeds the normal
effective-channel gain at those specific frequencies. This is **not** a
"true physical result of a frequency-selective array factor" in isolation;
more precisely, it is the joint result of:

- the frequency-selective **multipath channel** (`H_wideband`, which varies
  with subcarrier frequency because of differing path delays `tau`),
- the **cross-polarized element patterns** of the 128-port TX array,
- the **array responses** (steering behavior) of the selected DFT beam, and
- the **normal vs. sleep port excitation** (`w_normal` vs. the right-half-muted
  `w_sleep`).

Right-half array muting changes how the per-path/per-port contributions
combine coherently. Because that coherent combination is frequency-dependent
(different subcarriers see different relative phases between paths), muting
half the array can, on a small number of subcarriers, leave the *remaining*
active half's contributions combining more constructively than the full
array's contributions did — so `power_sleep > power_normal` at just those
frequencies.

**This does not mean sleep mode radiates more total power.** The sleep
codeword's total conducted power (`P_TX_SLEEP_W`, Section 19 and the strict
check in Section 24) remains exactly half of the normal codeword's total
power, independent of this per-subcarrier effective-channel effect.

In [ ]:
## 23. Plots

freq_mhz = active_frequencies_np / 1e6

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

ax = axes[0, 0]
ax.plot(freq_mhz, P_RX_NORMAL_DBM, label='normal', linewidth=0.8)
ax.plot(freq_mhz, P_RX_SLEEP_DBM, label='sleep', linewidth=0.8)
ax.set_xlabel('Subcarrier frequency offset [MHz]')
ax.set_ylabel('Received power [dBm]')
ax.set_title('Received power vs. subcarrier frequency')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[0, 1]
ax.plot(freq_mhz, SNR_NORMAL_DB, label='normal', linewidth=0.8)
ax.plot(freq_mhz, SNR_SLEEP_DB, label='sleep', linewidth=0.8)
ax.set_xlabel('Subcarrier frequency offset [MHz]')
ax.set_ylabel('SNR [dB]')
ax.set_title('SNR vs. subcarrier frequency')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1, 0]
ax.plot(freq_mhz, loss_db_per_sc, color='tab:red', linewidth=0.8)
ax.set_xlabel('Subcarrier frequency offset [MHz]')
ax.set_ylabel('Normal-sleep loss [dB]')
ax.set_title('Normal-sleep channel-power loss vs. subcarrier frequency')
ax.grid(True, alpha=0.3)

ax = axes[1, 1]
sorted_normal = np.sort(SNR_NORMAL_DB)
sorted_sleep = np.sort(SNR_SLEEP_DB)
cdf = np.linspace(0, 1, len(sorted_normal), endpoint=False)
ax.plot(sorted_normal, cdf, label='normal', linewidth=1.2)
ax.plot(sorted_sleep, cdf, label='sleep', linewidth=1.2)
ax.set_xlabel('SNR [dB]')
ax.set_ylabel('CDF')
ax.set_title('SNR CDF (normal vs. sleep)')
ax.legend()
ax.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()
print('Plots rendered.')


In [ ]:
## 24. Phase G1.1 summary assertions

assert H_wideband.shape == (NUM_ACTIVE_SUBCARRIERS, 2, 128)
assert len(active_frequencies_np) == NUM_ACTIVE_SUBCARRIERS
assert codebook.shape == (1024, 128)
assert valid_full_beam_count == valid_spatial_pmi_count * NUM_I2

assert 0 <= selected_pmi_wideband.i11 < NUM_HORIZONTAL_BEAMS
assert 0 <= selected_pmi_wideband.i12 < NUM_VERTICAL_BEAMS
assert 0 <= selected_pmi_wideband.i2 < NUM_I2

assert abs(norm_normal_wb - 1.0) < 1e-4
assert abs(norm_sleep_wb - 0.5) < 1e-4

# Strict sleep total-power check: P_TX_SLEEP_W must equal P_TX_NORMAL_W / 2 to
# within float32 propagation error (empirically ~7.7e-6 W absolute), not the
# much looser ~0.013 W tolerance used previously. Report actual errors before
# asserting; do NOT widen these tolerances if the assertion fails.
sleep_power_abs_err = abs(P_TX_SLEEP_W - P_TX_NORMAL_W / 2.0)
sleep_power_rel_err = sleep_power_abs_err / (P_TX_NORMAL_W / 2.0)
print(
    f'Sleep total-power check: P_TX_SLEEP_W={P_TX_SLEEP_W!r}, '
    f'P_TX_NORMAL_W/2={P_TX_NORMAL_W / 2.0!r}'
)
print(
    f'  absolute error = {sleep_power_abs_err:.6e} W, '
    f'relative error = {sleep_power_rel_err:.6e}'
)
assert math.isclose(
    P_TX_SLEEP_W,
    P_TX_NORMAL_W / 2.0,
    rel_tol=1e-6,
    abs_tol=1e-6,
), (
    f'Sleep total power {P_TX_SLEEP_W} W does not match half of normal total '
    f'power {P_TX_NORMAL_W / 2.0} W within rel_tol=1e-6, abs_tol=1e-6 '
    f'(absolute error={sleep_power_abs_err:.6e} W, '
    f'relative error={sleep_power_rel_err:.6e})'
)

assert np.isfinite(gain_normal).all() and (gain_normal >= 0).all()
assert np.isfinite(gain_sleep).all() and (gain_sleep >= 0).all()
assert np.isfinite(P_RX_NORMAL_W).all() and (P_RX_NORMAL_W >= 0).all()
assert np.isfinite(P_RX_SLEEP_W).all() and (P_RX_SLEEP_W >= 0).all()
assert np.isfinite(SNR_NORMAL).all() and (SNR_NORMAL >= 0).all()
assert np.isfinite(SNR_SLEEP).all() and (SNR_SLEEP >= 0).all()
assert np.array_equal(SINR_NORMAL, SNR_NORMAL)
assert np.array_equal(SINR_SLEEP, SNR_SLEEP)

# Strict center-frequency H consistency (re-verified independently of
# Section 17; see that section for the max-abs-error / max-rel-error /
# |H|-range diagnostics).
torch.testing.assert_close(H_center_t, H_t, rtol=1e-5, atol=1e-12)

# Both averaging conventions must be finite and must generally differ
# (log is nonlinear; see Section 14b and Section 22).
for name, lin_avg_db, db_mean in (
    ('rx_power (normal)', wideband_average_rx_power_dbm, mean_of_subcarrier_rx_power_dbm_normal),
    ('rx_power (sleep)', wideband_average_rx_power_dbm_sleep, mean_of_subcarrier_rx_power_dbm_sleep),
    ('snr (normal)', wideband_average_snr_db, mean_of_subcarrier_snr_db_normal),
    ('snr (sleep)', wideband_average_snr_db_sleep, mean_of_subcarrier_snr_db_sleep),
):
    assert math.isfinite(lin_avg_db) and math.isfinite(db_mean)

print('ALL PHASE G1.1 SUMMARY ASSERTIONS PASSED')
print(f'  wideband-selected PMI: {selected_pmi_wideband} (flat index {best_flat_idx_wideband})')
print(f'  matches G1 center-selected PMI: {matches_center_selected}')
print(f'  P_TX_NORMAL = {P_TX_NORMAL_W:.4f} W, P_TX_SLEEP = {P_TX_SLEEP_W:.4f} W')
print(
    f'  sleep-power absolute error = {sleep_power_abs_err:.6e} W, '
    f'relative error = {sleep_power_rel_err:.6e}'
)
print(f'  N_SC_W = {N_SC_W:.4e} W')
print(
    f'  mean_of_subcarrier_snr_db normal/sleep = {mean_of_subcarrier_snr_db_normal:.3f} / '
    f'{mean_of_subcarrier_snr_db_sleep:.3f}'
)
print(
    f'  wideband_average_snr_db normal/sleep = {wideband_average_snr_db:.3f} / '
    f'{wideband_average_snr_db_sleep:.3f}'
)
